In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

## Ingestion del archivo genre.csv

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .option("header", "true")
        .load(f"{bronze_folder_path}/genre.csv")
) 
display(genre_df)

genreId,genreName
12,Adventure
14,Fantasy
16,Animation
18,Drama
27,Horror
28,Action
35,Comedy
36,History
37,Western
53,Thriller


In [0]:
genre_schema = StructType([
    StructField("genreId", IntegerType(), True),
    StructField("genreName", StringType(), True)
])
genre_df = (
    spark.read
        .format('csv')
        .schema(genre_schema)
        .load(f"{bronze_folder_path}/genre.csv")
)
display(genre_df)
genre_df.printSchema()

genreId,genreName
null,genreName
12,Adventure
14,Fantasy
16,Animation
18,Drama
27,Horror
28,Action
35,Comedy
36,History
37,Western


root
 |-- genreId: integer (nullable = true)
 |-- genreName: string (nullable = true)



In [0]:
genre_df_final = (
    add_ingestion_date(genre_df)
    .select(
        col("genreId").alias("genre_id"),
        col("genreName").alias("genre_name"),
        lit(v_env).alias("env")
    )
)
display(genre_df_final)

genre_id,genre_name,env
null,genreName,dev
12,Adventure,dev
14,Fantasy,dev
16,Animation,dev
18,Drama,dev
27,Horror,dev
28,Action,dev
35,Comedy,dev
36,History,dev
37,Western,dev


In [0]:
(genre_df_final.write
    .format("parquet")
    .mode("overwrite")
    .save(f"{silver_folder_path}/genres")
)

In [0]:
dbutils.notebook.exit("Success")